# Cvičenie 6 - LSTM neural network

Na tomto cvičení sa oboznámite so základnými konceptami **rekurentných neurónových sietí**, konkrétne na príklade **LSTM siete**. Uvidíte tiež ako je možné s takýmito sieťami pracovať v Pytorchi, konkrétne na príklade siete predikujúcej nasledujúce slovo vo vete. Takúto sieť krok po kroku implementujete.

## Ciele

1. Oboznámiť sa s konceptom rekurentných neurónových sietí
2. Predstaviť si LSTM siete a vysvetliť ich fungovanie
3. Krok po kroku na vzorovej úlohe implementovať model využívajúci LSTM neurónovú sieť

## Zadanie

Naštudujte si základné koncepty fungovania rekurentných neurónových sietí a LSTM sietí konkrétne. Potom riešte úlohu, ktorej cieľom je vytvorenie LSTM siete, slúžiacej na generovanie krátkych vtipov. Úlohu riešte v jednotlivých krokoch, od komplexného spracovania dát, cez vytvorenie modelu siete, po jej natrénovanie.
___

## Krok 1 - motivácia pre rekurentné neurónové siete

Na doterajších cvičeniach ste si vyskúšali prácu s **feed-forward** neurónovými sieťami, od jednoduchých po komplexnejšie konvolučné neurónové siete. Oba predstavené príklady majú jednu spoločnú vlastnosť - dáta sieťou prechádzajú iba jedným smerom. Zjednodušene povedané, v sieti sa nenachádzajú žiadne cykly a výstup jednej predikcie nemá vplyv na budúce predikcie. To v predcházajúcich úlohách nebol problém, keďže v dátach, na ktoré bola aplikovaná predikcia, nebola žiadna sekvenčná závislosť. Avšak takéto siete nie sú vhodné pre problémy, kedy pracujeme so **sekvenciami dát**, napr. znaky v slovách, slová vo vete alebo hodnoty namerané v čase.

Presne pre takéto úlohy preto vznikli tzv. **rekurentné neurónové siete**. Takéto siete sú schopné udržiavať v pamäti informáciu o výstupe neurónov a používať ju pre nasledujúcu predikciu. Intuitívnym príkladom je napríklad **prediktívny text**, t.j. predpoveď nasledujúceho slova vo vete podľa zadanej celej sekvencie slov, nie len jednoho posledného slova. Dá sa teda povedať, že pracujú na základe **spätnej väzby** medzi výstupom zo siete pre budúce vstupy, preto sa im niekedy hovorí aj "feedback neural networks". Z tohto dôvodu sa rekurentné neurónové siete často používajú pre problémy **spracovania prirodzeného jazyka**. Takýto príklad si ukážeme aj na tomto cvičení.

Vizuálne je tento rozdiel možné zobraziť nasledovne:

<img src="./Images/rnn.png" style="width: 700px;"/>

___

## Krok 2 - LSTM sieť

**LSTM (long short-term memory)** sú typom rekurentnej neurónovej siete, ktorá je schopná učiť sa dlhodobé závislosti v sekvenčných údajoch. Tradičné RNN používajú jednoduché spätnoväzbové spojenia, čo im sťažuje uchovávanie informácií počas dlhého časového obdobia. To môže viesť k problémom miznúceho alebo explodujúceho gradientu v dlhších sekvenciách spracovávaných dát. LSTM siete sú navrhnuté tak, aby tento problém riešili a dosahujú to zavedením pamäťovej bunky, brán (vstupnej, pamäťovej, výstupnej) a skrytého stavu, ktoré riadia tok informácií do a zo siete, čo jej umožňuje selektívne uchovávať alebo vyraďovať informácie v priebehu času. Vďaka týmto vlastnostiam sú LSTM obzvlášť užitočné na úlohy, ako je spracovanie prirodzeného jazyka, rozpoznávanie reči a predpovedanie časových radov.

Základom LSTM sietí sú špeciálne neuróny LSTM bunky, ktorých štruktúra je zobrazená na nasledujúcom obrázku:

<img src="./Images/lstm.png" style="width: 400px;"/>

Keďže Pytorch ponúka triedu implementujúcu tieto bunky, nie je nutné detailne chápať ich presné fungovanie. Pre riešenie tejto úlohy je potrebné si ale všimnúť, ako vyzerajú vstupy a výstupy LSTM bunky. 

Vstupom do bunky t-teho kroku spracovania sekvencie sú:
1. **X<sub>t</sub>** - t-te dáta zo sekvencie
2. **h<sub>t-1</sub>** - minulý skrytý stav (krátkodobá pamäť)
3. **c<sub>t-1</sub>** - minulý bunkový stav (dlhodobá pamäť)

Výstupom sú:

2. **h<sub>t</sub>** - nový skrytý stav 
3. **c<sub>t</sub>** - nový bunkový stav 

___

## Krok 3 - praktická úloha

V tejto časti cvičenia krok po kroku implementujete sieť využívajúcu LSTM neurónovú sieť, ktorá bude schopná vykonávať predikciu nasledujúceho slova vo vete. Úlohu budete riešiť v nasledujúcich krokoch:

1. Importovanie potrebných knižníc
2. Prípadné použitie GPU paralelizmu
3. Načítanie a spracovanie dát pre trénovanie/testovanie
4. Budovanie datasetu
5. Definovanie modelu vašej LSTM siete
6. Trénovanie siete
7. Generovanie textu

### 3.1 Importy

Okrem už známych knižníc budeme používať najmä knižnicu *torchtext*, slúžiacu na prácu s prirodzeným jazykom, a knižnicu *re* pre prácu s regexami. 

> ##### ***Poznámka***
*torchtext nie je základnou súčasťou Pytorch-u, je potrebné ho dodatočne nainštalovať, nainštalujte si ho podľa [dokumentácie](https://pypi.org/project/torchtext/).*

##### *Spustiteľná ukážka*

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import re

from torch.utils.data import Dataset, DataLoader
from torchtext.data import get_tokenizer
from collections import Counter

___

### 3.2 CUDA

### Úloha č.1 

Implementuje overenie toho, či vaše zariadenie disponuje CUDA-enabled GPU. Ak áno, použite ho pri implementácii siete a jej trénovaní/testovaní.

In [ ]:
# your code goes here

___

### 3.3 Načítanie a spracovanie dát

Spracovanie dát je kľúčovým krokom v procese strojového učenia, častokrát dôležitejšie ako návrh samotného modelu. Vašou úlohou je načítať dateset zo súboru *Data/Jokes/jokes.txt*, obsahjúci 1622 krátkych vtipov, a následne ho spracovať. Spracovanie dát spočíva z nasledujúcich krokov:

1. **Čistenie dát** - odstránenie nechcených znakov atď.
2. **Tokenizácia** - rozdelenie viet na jednotlivé slová (tzv. tokeny)
3. **Vytvorenie slovnej zásoby** - zoznam unikátnych tokenov nášho textu
4. **Kódovanie tokenov** - mapovanie textových tokenov na celé čísla, s ktorými vie sieť pracovať

#### 3.3.1 Čistenie textu

### Úloha č.2

Implementujte funkciu *clean_text(text)*, ktorá vstupný text prevedie na malé písmená a odstráni z neho všetky znaky okrem alfanumerických znakov, nasledujúcich znakov: *?* *!* a medzier. Môžete pri tom použiť napríklad regulárne výrazy.

Príklad fungovanie funkcie:

*clean_text('Mac.ka 123!') = 'macka !'*


In [ ]:
def clean_text(text):
    # clean the text
    
    return text

print(clean_text("Mac.ka 123!"))

#### 3.3.2 Tokenizácia

Tokenizácia je proces delenia reťazcov obsahujúcich viaceré slová na zoznam jednotlivých slov. Keďže ide o jednu z najzákladnejších operácií pri spracovávaní prirodzeného jazyka, nemusíme ju implementovať sami, knižnica *torchtext* obsahuje viaceré tokenizačné funkcie. Ich fungovanie môžete vidieť na nasledujúcej ukážke:

##### *Spustiteľná ukážka*

In [ ]:
text =  "A random English sentence that will be split into tokens"

tokenizer = get_tokenizer("basic_english")
tokens = tokenizer(text)

print(tokens)

### Úloha č.3

Načítajte text zo súboru *jokes.txt*, iterujte ním po riadkoch pričom každý riadok vyčistite implementovanou funkciou a vyčistený text rozdeľte na tokeny a získaný zoznam uložte ich do poľa *all_jokes*.

> ##### ***Poznámka***
*Pozn: do poľa ukladajte tokeny jednotlivých vtipov vo vnorených poliach, nie len ich elementy*

In [ ]:
all_jokes = []

# your code goes here

#### 3.3.3 Vytvorenie slovnej zásoby

Vytváranie slovnej zásoby spočíva v nájdení všetkých unikátknych tokenov z nášho textu. Takýto zoznam nám vytvára akýsi slovník všetkých slov v texte, čo umožňuje napr. namapovanie slov na celé čísla. Spôsobov ako to realizovať je mnoho, jedným z nich je napríklad použitie dátovej štruktúry *Counter*. Jej fungovanie môžete vidieť na nasledujúcej ukážke:

##### *Spustiteľná ukážka*

In [ ]:
counter = Counter()

elements = ["a", "b", "c", "d"]
counter.update(elements)

print(counter)

new_elements = ["a", "a", "a", "e"]
counter.update(new_elements)

print(counter)

unique = list(counter)
print("Unique values:", unique)

Ako môžete vidieť, counter vytvára slovník obsahujúci jednotlivé elementy a počet koľkokrát boli do counteru pridané. Následne je možné získať zoznam unikátnych elementov konvertovaním slovníka na pole.

### Úloha č.4

Kód predchádzajúcej úlohy rozšírte o vytvorenie counter-u, ktorý iteratívne rozširujte o tokeny jednotlivých vtipov. Po načítaní všetkých vtipov získajte zoznam unikátnych tokenov všetkých vtipov do poľa *unique_tokens*. Ten bude predstavovať vašu slovnú zásobu.

In [ ]:
# your code goes here

#### 3.3.4 Kódovanie tokenov

Ako už viete, neurónové siete v svojej podstate pracujú s číselnými hodnotami, nie reťazcami znakov. Preto je potrebné vami vygenerovaný zoznam tokenov jednotlivých vtipov namapovať na číselné hodnoty, čo vám umožní práve vytvorená slovná zásoba. Vďaka nej môžete tokeny jednoducho namapovať na celé čísla, pričom výsledná hodnota slova bude index, na ktorom sa nachádza v slovnej zásobe.

### Úloha č.5

Kód z predchádzajúcej úlohy rozšírte o vytvorenie nasledujúcich dvoch slovníkov:
1. **token_to_index str->int** - mapuje tokeny na čísla
2. **index_to_token int->str** - mapuje čísla na tokeny

Môže vám pri tom poslúžiť napríklad metóda *enumerate()*

Po vytvorení týchto slovníkov namapujte dáta v poli *all_jokes* na ich číselné reprezentácie. Nezabudnite však udržať pôvodnú 2D štruktúru poľa.

In [ ]:
# your code goes here

___

### 3.4 Budovanie datasetu

V tomto bode už máte dáta vyčistené a spracované, ostáva už len vytvoriť objekty *Dataset* a *Dataloader*, s ktorými budete pracovať pri trénovaní siete. Budovanie vlastných datasetov ste si už vyskúšali na cvičení císlo 2. Pre zopakovanie, je potrebné vytvoriť vlastnú triedu rozširujúcu triedu *Dataset* a v nej implementovať nasledujúce metódy:

1. **__init__(self)** - konštruktor, inicializujeme v ňom štruktúru obsahujúcu naše dáta
2. **__len__(self)** - metóda, ktorá vracia celkový počet vzoriek v datasete
3. **__getitem__(self, idx)** - metóda, ktorá vracia dvojicu **feature, label** dátovej vzorky na indexe **idx**

Je však potrebné si uvedomiť, akú štruktúru budú mať naše *features (X), label (Y)*. Keďže pracujete so sekvenčnými dátami, X bude sekvencia po sebe idúcich slov a Y bude nasledujúce slovo zo sekvencie. Musíte si teda určiť dĺžku sekvencií, s ktorými budete pracovať a všetky vtipy rozbiť na sekvencie slov danej dĺžky. Štruktúru datasetu môžte teda vidieť na nasledujúcej ilustrácii, pri zvolenej dĺžke sekvencie 4 (t.j., 4 spracovávané slová):

**If life gives you melons, you might have dyslexia.** <br/>
dĺžka sekvencie = 4

| **X** | **Y** |
|-------|--------|
| If, life, gives, you | melons |
| life, gives, you, melons | you |
| gives, you, melons, you | might |
| you, melons, you, might | have |
| melons, you, might, have | dyslexia |

Ak takto spracujete všetky vtipy z datasetu, získate tak rozsiahly zoznam sekvencií, na ktorých možete sieť trénovať. Samozreje, pracovať budete so slovami mapovanými na čísla, ukážka obsahuje celé slová len pre ilustráciu.

### Úloha č.6

Implementuje triedu *JokesDataset*, ktorá spracuje dáta z poľa *all_jokes* podľa hore ukázaného princípu, s vami určenou dĺžkou sekvencie slov. Následne vytvorte pomocou nej objekt *Dataloader*.

**Pozor, dáta pri spracovaní ukladajte do datasetu ako tensory, nie polia!**

In [ ]:
# implement dataset
class JokesDataset(Dataset):
    def __init__(self, sequence_length, jokes):
        # process data into sequences and lebels
    
    def __len__(self):
        # return lenght of data
        
    def __getitem__(self, index):
        # return data point
        
# define hyperparameters
batch_size =  
seq_len = 

# initialize dataloader

___

### 3.5 Definovanie modelu siete

Keď už máte pripravené dáta, na ktorých budete sieť trénovať, ďalším krokom je definovanie jej modelu. Keďže pracujete so sekvenčnými dátami (sekvencie tokenov), je zrejmé, že váš model musí obsahovať aspoň jednu rekurentnú vrstvu. V tomto prípade použijete konkrétne LSTM vrstvu, ktorá je v Pytorchi už implementovaná. Model, ktorý budete vytvárať, bude mať teda nasledujúcu štruktúru:

1. **Embedding vrstva** - mapovanie číselnej rerezentácie tokenov na tensory s danou dĺžkou
2. **LSTM vrstva** - vrstva zložená z LSTM buniek, ako boli opísané vyššie
3. **Lineárna vrstva** - vykonáva klasifikáciu (t.j. predikciu nasledujúceho slova sekvencie), na základe výstupu z LSTM vrstvy

#### 3.5.1 Embedding

Proces embeddingu spočíva v mapovaní vstupov (v našom prípade tokenov reprezentovaných celým číslom), do viacrozmerného priestoru. Často sa využíva pri spracovaní prirodezeného jazyka pre reprezentáciu slov resp. znakov namiesto one-hot encodingu. Pre príklad, ak váš dataset obsahuje 10 000 rôznych slov, one-hot encoding každé mapuje na tensory veľkosti 10 000, čo vedie k veľmi rozsiahlym modelom. Práve embedding dokáže ľubovoľný počet tokenov namapovať na nami zvolený rozmer.

Embedding vrstva je v Pytorchi implementovaná triedou *nn.Embedding*, ktorej fungovanie môžete vidieť v nasledujúcej ukážke:

##### *Spustiteľná ukážka*

In [ ]:
num_embeddings = 10 # the number of unique values to embed
embedding_dim = 3 # the size of the embedded output

embedding = nn.Embedding(num_embeddings, embedding_dim)

for i in range(num_embeddings):
    inp = torch.Tensor([i]).int()
    embedded = embedding(inp)
    
    print(f"{inp} was embedded into {embedded}")

Ako môžete vidieť, embedding namapoval hodnoty na rôzne tensory o veľkosti 3, pričom one-hot encoding vytvoril tensory o veľkosti 10.

#### 3.5.2 LSTM vrstva

Ako bolo spomínané vyššie, Pytorch obsahuje implementáciu LSTM vrstiev. Pre jej použitie je však potrebné chápať niektoré jej vybrané parametre, vstupy a výstupy.

**Paremetre:**
1. **input_size** - počet vlastností vstupu x
2. **hidden_size** - veľkosť skrytého stavu h (viď. krok 2)
3. **batch_first** - ak je *True*, potom sa vstupné a výstupné tenzory poskytujú v tvare **(batch, seq, feature)** namiesto **(seq, batch, feature)**, pričom:
    - **batch** - počet vzoriek v jednej dávke
    - **seq** - dĺžka vstupnej sekvencie dát
    - **feature** - počet vlastností dátových vzoriek (t.j. zhodné s **input_size**)
    
**Vstupy: input, (h_0, c_0)**
1. **input** - tenzor tvaru **(seq, feature)** pre vstup bez dávky, **(seq, batch, feature)**, keď *batch_first=False*, alebo **(batch, seq, feature)**, keď *batch_first=True*, obsahujúci vlastnosti vstupnej sekvencie.
2. **h_0** - tensor obsahujúci počiatočný skrytý stav (nuly, ak nie je uvedený)
2. **c_0** - tensor obsahujúci počiatočný bunkový stav (nuly, ak nie je uvedený)

**Výstupy: output, (h_n, c_n)**
1. **output** - tenzor tvaru **(seq, feature)** pre vstup bez dávky, **(seq, batch, feature)**, keď *batch_first=False*, alebo **(batch, seq, feature)**, keď *batch_first=True*, obsahujúca výstupné vlastnosti (h_t), pre každé t sekvenčného spracovania (viď. krok 2)
2. **h_n** -  tensor obsahujúci výsledný skrytý stav pre posledný krok sekvencie
3. **c_n** - tensor obsahujúci výsledný bunkový stav pre posledný krok sekvencie

Pre hlbšie pochopenie tejto triedy si môžete prečítať jej [dokumentáciu](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html).

#### 3.5.3 Model siete

Výsledný model siete je možné vybudovať z hore spomenutých vrstiev a jednej lineárnej vrstvy vykonávajúcej klasifikáciu. Nižšie môžete vidieť implementáciu takejto siete. Oboznámte s jej štruktúrou, prípadne ju môžete upraviť (pridanie vrstiev, úprava parametrov atď.) podľa seba.

##### *Spustiteľná ukážka*

In [ ]:
class lstm(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, embed_size) # embedding layer 
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True) # lstm layer
        self.linear = nn.Linear(hidden_size,vocab_size) # classification layer
        
    def forward(self, x):
        # input sequence to embeddings (vocab_size tokens to embed_size tensors)
        embedded = self.embed(x)       
                
        # pass the embedded sequence to the lstm layer
        output, _ = self.lstm(embedded)
                
        # only do classification o the last h_t
        # output is of size (batch, seq, feature), so take the last slice on second dimension
        output = output[:,-1,:]
        output = torch.relu(output)
        
        # classify through FC
        output = self.linear(output)
        return output

Výstupom predikcie je teda tensor o veľkosti *vocab_size*, pretože klasfikáciu vykonávame pre *vobca_size* tried (t.j. každé slovo slovnej zásoby je triedou).

___

### 3.6 Trénovanie siete

### Úloha č.7 

Implementuje trénovaciu slučke pre vami vytovernú sieť. Stratovú funkciu, optimizátor ako aj hyperparametre trénovania si zvoľte sami a sieť trénujte na dátach z vami iplmentovaného JokesDaset setu. Dbajte na vhodnú veľkosť tensorov vstupujúcu do siete.

In [ ]:
# your code goes here

___

### 3.7 Generovanie textu

### Úloha č.8

Implementujte funkciu, ktorá na vstupe dostáva:
1. **seed_text** - vstupný text (začiatočné slová generovaného vtipu)
2. **words_to_generate** - počet slov na dogenerovanie
3. **sequence_len** - dľžka vstupnej sequencie do siete
4. **model** - váš predtrénovaný model

a vykonáva generovanie nového vtipu pomocou natrénovaného modelu.

Princíp fungovania funkcie je nasledovný:
1. Spracovať vstupný text na pole indexov
2. Iteratívne pre počet nových slov: <br/>
    2.1 Z posledných *sequence_len* hodnôt v poli predikovať nasledujúce slovo <br/>
    2.2 Predikované slovo pridať do poľa <br/>
3. Hodnoty vo výslednom poli previesť na ich textovú reprezentáciu

In [ ]:
def generate_joke(seed_text, words_to_generate, model, sequence_len):
    # your code goes here

joke = generate_joke("Knock knock who's there?", 10, model, seq_len)
print(joke)